In [0]:
# ============================================================
# NOTEBOOK 04 — ML Training with PySpark MLlib
# ============================================================
# Reads Silver Parquet, builds a Spark ML Pipeline with
# StringIndexer + VectorAssembler + RandomForestClassifier,
# trains a delay prediction model, and evaluates accuracy.
# ============================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

SILVER_PATH = "/Volumes/workspace/default/flight_raw_data/silver"
ML_PATH = "/Volumes/workspace/default/flight_raw_data/ml"

print("Silver input path:", SILVER_PATH)
print("ML output path:", ML_PATH)

Silver input path: /Volumes/workspace/default/flight_raw_data/silver
ML output path: /Volumes/workspace/default/flight_raw_data/ml


In [0]:
# ============================================================
# Step 1 — Read Silver Parquet and select ML features
# ============================================================
# We select only columns that would realistically be available
# BEFORE a flight departs — we cannot use ARRIVAL_DELAY,
# ELAPSED_TIME, or AIR_TIME as features because those are only
# known after the flight lands. Using post-flight data to predict
# delays would be data leakage — a critical ML mistake.
#
# Features we use (all known before departure):
# - MONTH, DAY, DAY_OF_WEEK: time-based patterns
# - AIRLINE: carrier performance history
# - DEPARTURE_DELAY: known at departure gate
# - DISTANCE: flight length
# - SCHEDULED_DEPARTURE: time of day
# - TAXI_OUT: ground delay before takeoff
# ============================================================

df_silver = spark.read.parquet(f"{SILVER_PATH}/flights")

df_ml = df_silver.select(
    "MONTH",
    "DAY_OF_WEEK",
    "AIRLINE",
    "DEPARTURE_DELAY",
    "DISTANCE",
    "SCHEDULED_DEPARTURE",
    "TAXI_OUT",
    "IS_DELAYED"
).dropna()

print("ML dataset row count:", df_ml.count())
df_ml.show(5)

ML dataset row count: 5714008
+-----+-----------+-------+---------------+--------+-------------------+--------+----------+
|MONTH|DAY_OF_WEEK|AIRLINE|DEPARTURE_DELAY|DISTANCE|SCHEDULED_DEPARTURE|TAXI_OUT|IS_DELAYED|
+-----+-----------+-------+---------------+--------+-------------------+--------+----------+
|    7|          5|     AS|            -14|    1180|               1620|      11|         0|
|    7|          5|     B6|             28|    1237|               1620|      12|         1|
|    7|          5|     B6|             -3|    1188|               1620|       9|         0|
|    7|          5|     AA|             -9|    1045|               1620|      10|         0|
|    7|          5|     AA|             -2|    1773|               1620|      14|         0|
+-----+-----------+-------+---------------+--------+-------------------+--------+----------+
only showing top 5 rows


In [0]:
# ============================================================
# Step 2 — Split into training and test sets
# ============================================================
# We split 80% of data for training and 20% for testing.
# seed=42 ensures the split is reproducible — running this
# notebook again will always produce the same split.
# We stratify by IS_DELAYED to maintain the class ratio.
# ============================================================

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

print("Training delay distribution:")
train_df.groupBy("IS_DELAYED").count().show()

Training rows: 4571191
Test rows: 1142817
Training delay distribution:
+----------+-------+
|IS_DELAYED|  count|
+----------+-------+
|         1| 819012|
|         0|3752179|
+----------+-------+



In [0]:
# ============================================================
# Step 3 — Build the Spark ML Pipeline
# ============================================================
# A Spark ML Pipeline chains multiple stages together so they
# execute in sequence automatically during training and testing.
#
# Stage 1 — StringIndexer:
#   Converts AIRLINE text column (e.g. "AA", "DL") into numeric
#   indices (e.g. 0.0, 1.0, 2.0). MLlib requires all inputs
#   to be numeric — it cannot work with raw strings.
#
# Stage 2 — VectorAssembler:
#   Combines all feature columns into a single column called
#   "features" containing a dense vector. This is the mandatory
#   input format for all MLlib classifiers.
#
# Stage 3 — RandomForestClassifier:
#   Trains 50 decision trees and combines their predictions.
#   numTrees=50 balances accuracy with training speed.
#   maxDepth=10 controls how deep each tree can grow.
#   labelCol="IS_DELAYED" is what we are predicting.
# ============================================================

# Stage 1: Convert AIRLINE string to numeric index
airline_indexer = StringIndexer(
    inputCol="AIRLINE",
    outputCol="AIRLINE_INDEX",
    handleInvalid="keep"
)

# Stage 2: Assemble all features into a single vector
assembler = VectorAssembler(
    inputCols=[
        "MONTH",
        "DAY_OF_WEEK",
        "AIRLINE_INDEX",
        "DEPARTURE_DELAY",
        "DISTANCE",
        "SCHEDULED_DEPARTURE",
        "TAXI_OUT"
    ],
    outputCol="features"
)

# Stage 3: Random Forest Classifier
rf = RandomForestClassifier(
    labelCol="IS_DELAYED",
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    seed=42
)

# Chain all stages into a single Pipeline
pipeline = Pipeline(stages=[airline_indexer, assembler, rf])

print("Pipeline built with stages:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  Stage {i+1}: {type(stage).__name__}")

Pipeline built with stages:
  Stage 1: StringIndexer
  Stage 2: VectorAssembler
  Stage 3: RandomForestClassifier


In [0]:
# ============================================================
# Step 4 — Train the Pipeline
# ============================================================
# pipeline.fit() runs all three stages in sequence:
# 1. StringIndexer learns the AIRLINE → index mapping
# 2. VectorAssembler combines columns into feature vectors
# 3. RandomForestClassifier trains 50 decision trees
# This is the most computationally expensive step.
# ============================================================

print("Training started — this will take a few minutes...")

model = pipeline.fit(train_df)

print("Training complete.")

Training started — this will take a few minutes...
Training complete.


In [0]:
# ============================================================
# Step 5 — Evaluate the model on test data
# ============================================================
# model.transform() runs the test data through all pipeline
# stages and produces predictions.
#
# We evaluate using three metrics:
# - Accuracy: percentage of correct predictions overall
# - AUC (Area Under ROC Curve): measures how well the model
#   separates delayed vs non-delayed flights. 1.0 is perfect,
#   0.5 is random guessing. Industry standard ML metric.
# - F1 Score: harmonic mean of precision and recall — important
#   for imbalanced datasets like ours where delayed flights are
#   only 18% of the data.
# ============================================================

predictions = model.transform(test_df)

# Accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="IS_DELAYED",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = accuracy_evaluator.evaluate(predictions)

# AUC
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="IS_DELAYED",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = auc_evaluator.evaluate(predictions)

# F1 Score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="IS_DELAYED",
    predictionCol="prediction",
    metricName="f1"
)
f1 = f1_evaluator.evaluate(predictions)

print("=== Model Evaluation Results ===")
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"AUC:       {auc:.4f}")
print(f"F1 Score:  {f1:.4f}")

=== Model Evaluation Results ===
Accuracy:  0.9444 (94.44%)
AUC:       0.9610
F1 Score:  0.9428


In [0]:
# ============================================================
# Step 6 — Feature Importance
# ============================================================
# Random Forest calculates how much each feature contributed
# to reducing prediction error across all 50 trees.
# Higher importance = stronger predictor of delays.
# This is one of the most valuable outputs for business insight
# — it tells you WHAT causes delays, not just whether a flight
# will be delayed.
# ============================================================

rf_model = model.stages[-1]  # Extract the trained RF model

feature_names = [
    "MONTH",
    "DAY_OF_WEEK",
    "AIRLINE_INDEX",
    "DEPARTURE_DELAY",
    "DISTANCE",
    "SCHEDULED_DEPARTURE",
    "TAXI_OUT"
]

importances = rf_model.featureImportances.toArray()

importance_data = sorted(
    zip(feature_names, importances),
    key=lambda x: x[1],
    reverse=True
)

print("=== Feature Importance Rankings ===")
print(f"{'Feature':<25} {'Importance':>10}")
print("-" * 37)
for feature, importance in importance_data:
    bar = "█" * int(importance * 100)
    print(f"{feature:<25} {importance:>10.4f}  {bar}")

=== Feature Importance Rankings ===
Feature                   Importance
-------------------------------------
DEPARTURE_DELAY               0.8907  █████████████████████████████████████████████████████████████████████████████████████████
TAXI_OUT                      0.0875  ████████
SCHEDULED_DEPARTURE           0.0095  
DISTANCE                      0.0061  
AIRLINE_INDEX                 0.0042  
MONTH                         0.0018  
DAY_OF_WEEK                   0.0001  


In [0]:
# ============================================================
# Step 7 — Save ML results as Parquet for the dashboard
# ============================================================

import pandas as pd

# Save evaluation metrics as a single-row Parquet
metrics_data = [(accuracy, auc, f1, 5714008, 50, 10)]
metrics_df = spark.createDataFrame(
    metrics_data,
    ["ACCURACY", "AUC", "F1_SCORE", "TRAINING_ROWS", "NUM_TREES", "MAX_DEPTH"]
)
metrics_df.write.mode("overwrite").parquet(f"{ML_PATH}/metrics")

# Save feature importances as Parquet
importance_rows = [(f, float(i)) for f, i in importance_data]
importance_df = spark.createDataFrame(
    importance_rows,
    ["FEATURE", "IMPORTANCE"]
)
importance_df.write.mode("overwrite").parquet(f"{ML_PATH}/feature_importance")

print("ML results saved.")
print("Metrics:")
metrics_df.show()
print("Feature Importances:")
importance_df.show()

ML results saved.
Metrics:
+------------------+------------------+------------------+-------------+---------+---------+
|          ACCURACY|               AUC|          F1_SCORE|TRAINING_ROWS|NUM_TREES|MAX_DEPTH|
+------------------+------------------+------------------+-------------+---------+---------+
|0.9444241728990731|0.9609530242019299|0.9428437267455051|      5714008|       50|       10|
+------------------+------------------+------------------+-------------+---------+---------+

Feature Importances:
+-------------------+--------------------+
|            FEATURE|          IMPORTANCE|
+-------------------+--------------------+
|    DEPARTURE_DELAY|  0.8907350965952195|
|           TAXI_OUT| 0.08753094226472287|
|SCHEDULED_DEPARTURE|0.009467803034877893|
|           DISTANCE|0.006136760254549812|
|      AIRLINE_INDEX|0.004217431148597981|
|              MONTH|0.001790115795920...|
|        DAY_OF_WEEK|1.218509061116710...|
+-------------------+--------------------+

